In [ ]:
# load required libraries
import sys
sys.path.append('../utils')
# data model - Reading annotated data (by annotators)
# the data model class below returns masks as well
from json_parser import CellMaskDataset
import os
from PIL import Image
from IPython.display import display
import numpy as np
import cv2
import pandas as pd
from typing import List, Union, Dict, Final, Tuple, Optional

In [ ]:
MAX_IMAGE_SIDE: int = 4512
CROP_SIZE: int = 96

DATASET_PATHS: List[str] = [
    '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_caged',
    '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240228_k562_10x_caged',
    '/home/cellareye/Cellanome/Data/20240228_k562_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240228_nk92_10x_caged',
    '/home/cellareye/Cellanome/Data/20240228_nk92_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240425_nk92_10x_caged',
    '/home/cellareye/Cellanome/Data/20240425_nk92_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240228_hela-suspension_10x_caged',
    '/home/cellareye/Cellanome/Data/20240228_hela-suspension_10x_uncaged', 
    '/home/cellareye/Cellanome/Data/20240314_imr90-suspension_10x_caged', 
    '/home/cellareye/Cellanome/Data/20240314_imr90-suspension_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240307_pbmc-beads_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_10x_caged',
    '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240306_mousepbmc-beads_10x_caged',
    '/home/cellareye/Cellanome/Data/20240306_mousepbmc-beads_10x_uncaged',
    '/home/cellareye/Cellanome/Data/20240306_mousepbmc-nobeads_10x_caged',
    '/home/cellareye/Cellanome/Data/20240306_mousepbmc-nobeads_10x_uncaged', 
]

# sizes of the images in each dataset 
# NOTE: datasets with mixed image sizes are not supported
DATASET_SIZES: List[Tuple[int, int]] = [(4512, 4512)] * len(DATASET_PATHS)

OUTPUT_FOLDER = 'cell_classification_data_in_focus'
if not os.path.exists(OUTPUT_FOLDER):
    os.mkdir(OUTPUT_FOLDER)

ONLY_USE_BEST_FOCUS_IMAGE: bool = True

def extract_cell_type(dataset_name):
    for cell_type in CELL_TYPES:
        if cell_type in dataset_name.lower():
            return cell_type
    return 'unknown'

In [ ]:
CELL_TYPES = ['jurkat', 'k562', 'nk92', 'imr90', 'hela', 'mousepbmc', 'pbmc', 'hs675t', 'neuron']
CELL_STATES: List[str] = ['suspension', 'adhered']

# classnames of interest in annotations, and how they are mapped to an ID
ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP: Dict[str, int] = {'cell': 0, 'Cell': 0, 'dead-cell': 0, 'dying/dead cells': 0, 
                                                            'cytoplasm': 1,  'cell-adhered': 1, 
                                                            'soma': 2, 
                                                            'bead': 3, 'Bead': 3,
                                                            # 'nucleus': 4, # we do not need to classify nuclei, so no need to read the annotations 
                                                            # 'cage': 5, 'cages': 5, # we do not need to classify cages, so no need to read the annotations 
                                                           }
# cell state for cell class ids included in the annotations
ANNOTATIONS_CLASS_IDS_TO_CELL_STATE_MAP: Dict[int, str] = {0: 'suspension', 1: 'adhered', 2: 'adhered'}

# class IDs and their corresponding class names (conventional) for non-cell objects in the annotations that we are going to classify
ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP: Dict[int, str] = {3: 'bead'}

annotations_class_ids: List[int] = list(set(ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP.values()))
covered_class_ids: List[int] = list(ANNOTATIONS_CLASS_IDS_TO_CELL_STATE_MAP.keys()) + list(ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP.keys())
if len([i for i in annotations_class_ids if i not in covered_class_ids]) > 0 or len([i for i in covered_class_ids if i not in annotations_class_ids]) > 0:
    print(f"[WARN]: ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP should cover all the remaining class IDs in the annotations" 
          f" that are not listed in ANNOTATIONS_CLASS_IDS_TO_CELL_STATE_MAP")


CLASSIFIER_LABEL_MAP: Dict[int, Tuple[str, str]] = {}
STARTING_CLASS_INDEX: int = 0

class_id: int = STARTING_CLASS_INDEX
for cell_state in CELL_STATES:
    for cell_type in CELL_TYPES:
        CLASSIFIER_LABEL_MAP[class_id] = (cell_type,  cell_state) 
        class_id += 1

for cls_id, cls_name in ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP.items():
    CLASSIFIER_LABEL_MAP[class_id] = (cls_name,  'na') 
    class_id += 1

print(f"[INFO] Number of classed for cell type classification: {class_id}")

## Create cropped images of different classes
Skip this step onces the training images are created. This step should be repeated for any additional dataset/cell type available. 

In [ ]:
def create_dataset_classes(dataset_path: str, 
                           class_names_to_class_ids_map: Dict[str, int], 
                           train: bool=True, 
                           percentage_to_expand_bbox_boundaries: float=0.0, 
                           max_larger_side: int=MAX_IMAGE_SIDE, 
                           max_smaller_side: int=MAX_IMAGE_SIDE):
    images_path:str = dataset_path
    annotations_path: str = os.path.join(dataset_path, 'annotations')

    annotations_images_map: pd.DataFrame = pd.read_csv(os.path.join(dataset_path, 'annotation_images_mapping.csv'))

    test_files: List[str] = []
    train_files: List[str] = []
    
    with open(os.path.join(dataset_path, 'test.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        test_files += filenames

    with open(os.path.join(dataset_path, 'train.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        train_files += filenames

 
    columns: List[str] = list(annotations_images_map.columns)
    image_columns = [column_name for column_name in columns if 'white_dz' in column_name.lower()]

    if len(image_columns) > 0 and ONLY_USE_BEST_FOCUS_IMAGE:
        image_columns = [column_name for column_name in columns if 'white_dz0' in column_name.lower()]
        
    if len(image_columns) == 0:
        # this is not a focus sweep dataset, get the BF image
        for column_name in columns:
            if 'bf' in column_name.lower() or 'white' in column_name.lower():
                image_columns = [column_name]
                break

    if len(image_columns) == 0:
        print('[ERROR]: No brightfield image folder could be extracted from annotation_images_mapping.csv file for the dataset')
    
    train_map_dict: Dict[str, List[str]] = {}
    test_map_dict: Dict[str, List[str]] = {}
    for _, row in annotations_images_map.iterrows():
        annotations_filename = row['annotation_json']
        name = '.'.join(annotations_filename.strip().split('.')[:-1])
        if name in test_files:
            test_map_dict[annotations_filename] = list(row[image_columns])
        else:
            train_map_dict[annotations_filename] = list(row[image_columns])


    # datasets
    if train:
        dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                  annotations=train_map_dict,
                                  max_images_to_consider_for_each_annotation=1, # in case of focus sweep dataset, only pick 1 image randomly
                                  labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                  percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                  color_depth=8, 
                                  min_object_diameter = 6.0,
                                  scale_factor_dict={}, 
                                  max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                  normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    else:
        dataset = CellMaskDataset(images_path=images_path, annotations_path=annotations_path, 
                                  annotations=test_map_dict,
                                  max_images_to_consider_for_each_annotation=1,
                                  labels_of_interest=list(class_names_to_class_ids_map.keys()), 
                                  percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries, 
                                  color_depth=8, 
                                  min_object_diameter = 6.0,
                                  scale_factor_dict={}, 
                                  max_larger_side = max_larger_side, max_smaller_side = max_smaller_side,
                                  normalize=False, class_names_to_ids_map=class_names_to_class_ids_map)

    return dataset

In [ ]:
def create_training_images(dataset_path: str, 
                           output_base_folder: str, 
                           crop_size: int, 
                           percentage_to_expand_bbox_boundaries: float,
                           classifier_label_map: Dict[int, Tuple[str, str]], 
                           train: bool=True):
    
    reverse_label_map: Dict[Tuple[str, str], int] = {v: k for k, v in classifier_label_map.items()}
    
    dataset_name: str = os.path.basename(dataset_path)
    dataset_cell_type: str = extract_cell_type(dataset_name)

    print(f"[INFO] Processing {dataset_name} ...")
    print(f"[INFO] Cell type in the dataset: {dataset_cell_type}")
    if dataset_cell_type.lower() == 'unknown':
        print("[ERROR] The cell type in the dataset is Unknown! Cannot proceed further! Existing ... ")
    
    half_crop_size: int = int(np.ceil(crop_size / 2.0))
    
    dataset = create_dataset_classes(dataset_path=dataset_path, 
                                     class_names_to_class_ids_map=ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP, 
                                     train=train,
                                     percentage_to_expand_bbox_boundaries=percentage_to_expand_bbox_boundaries)

     # save the results in the output folders
    if train:
        output_folder: str = os.path.join(output_base_folder, 'train')
    else:
        output_folder: str = os.path.join(output_base_folder, 'test')
            
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)
    
    for idx in range(len(dataset)):
        # read the sample 
        sample = dataset[idx]

        img: np.ndarray = sample['image']

        img_height, img_width = img.shape[:2]
    
        num_channels: int = 1
        if len(img.shape) > 2:
            num_channels = img.shape[2]
        
        img_name: str = '.'.join(sample['name'].strip().split('.')[:-1])
        obj_count: int = 0

        for _, row in sample['annotations'].iterrows():
            # box coordinates
            xtl, ytl, xbr, ybr = row[['xtl', 'ytl', 'xbr', 'ybr']].values.astype(int)
            label = row['label']
            
            obj_center_x: int = int(np.round(xtl + xbr) / 2.0)
            obj_center_y: int = int(np.round(ytl + ybr) / 2.0)
            half_obj_size: int = int(max(np.ceil((xbr - xtl) / 2.0), np.ceil((ybr - ytl) / 2.0)))

            # form an sqaure crop around the object and centered around the center of the box
            # note that the bounding box of the object is already expanded by the factor percentage_to_expand_bbox_boundaries
            # passed, so no need to further expand here to have some margin around the object
            start_pixel_in_x: int = max(0, obj_center_x - half_obj_size)
            end_pixel_in_x: int = start_pixel_in_x + 2 * half_obj_size

            # adjust for the objects on the image boundaries
            if end_pixel_in_x >= img_width:
                end_pixel_in_x = img_width
                start_pixel_in_x = img_width - 2 * half_obj_size

            start_pixel_in_y: int = max(0, obj_center_y - half_obj_size)
            end_pixel_in_y: int = start_pixel_in_y + 2 * half_obj_size

            if end_pixel_in_y >= img_height:
                end_pixel_in_y = img_height
                start_pixel_in_y = img_height - 2 * half_obj_size

            # resize the object within the provided crop size, we do that to ensure the crop only 
            # contains the object class as annotated
            if half_obj_size > half_crop_size:
                interpolation_scheme = cv2.INTER_AREA
            else:
                interpolation_scheme = cv2.INTER_CUBIC
            
            cropped_img: np.ndarray = cv2.resize(img[start_pixel_in_y:end_pixel_in_y, start_pixel_in_x:end_pixel_in_x], 
                                                 (crop_size, crop_size), 
                                                 interpolation_scheme)

            if label in ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP:
                class_id: int = reverse_label_map[(ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP[label], 'na')]
            else:
                class_id: int = reverse_label_map[(dataset_cell_type, ANNOTATIONS_CLASS_IDS_TO_CELL_STATE_MAP[label])]

            output_class_folder: str = os.path.join(output_folder, str(class_id))
            if not os.path.exists(output_class_folder):
                os.mkdir(output_class_folder)
                
            cv2.imwrite(os.path.join(output_class_folder, img_name + '_' + str(obj_count) + '.jpg'), 
                        cropped_img, [int(cv2.IMWRITE_JPEG_QUALITY), 100])

            obj_count += 1

In [ ]:
### SKIP this step if training images are already created
for dataset_path in DATASET_PATHS:
    create_training_images(dataset_path=dataset_path, 
                           output_base_folder=OUTPUT_FOLDER, 
                           crop_size=CROP_SIZE, 
                           percentage_to_expand_bbox_boundaries=0.15,
                           classifier_label_map=CLASSIFIER_LABEL_MAP, 
                           train=True)
    create_training_images(dataset_path=dataset_path, 
                           output_base_folder=OUTPUT_FOLDER, 
                           crop_size=CROP_SIZE, 
                           percentage_to_expand_bbox_boundaries=0.15,
                           classifier_label_map=CLASSIFIER_LABEL_MAP, 
                           train=False)

## Create the training set using Mask R-CNN embeddings

In [ ]:
from mask_rcnn_model import MaskRCNNInstanceSegmentation, CROP_CORNERS, RESIZE, MODEL_WEIGHTS_PATH
from cv_utils import overlap_batch
import torch
detector = MaskRCNNInstanceSegmentation(weights_path=MODEL_WEIGHTS_PATH)

In [ ]:
def assign_boxes_to_crops(roi_boxes: np.ndarray, crop_corners: np.ndarray):
    """
    A function to assign each RoI bounding box to a sub-image as specified by overlapping crops in crop_corners. Each 
    box is assigned to the cropped sub-image with the maximum overlap (maximum intersection).
    Args:
        - roi_boxes (a float numpy array): An array of N by 4 bounding boxes for N bounding boxes across the image
        - crop_corners (a float numpy array): An array of M by 4 coordinates for M overlapping crops covering the image resolution
    Returns:
        - An (N, ) array of integers with i-th element the index of the crop corner that the i-th box is assigend to.
    """
    if len(roi_boxes) == 0:
        return np.zeros((0, ), dtype=int)

    if len(crop_corners) == 0:
        return np.zeros((len(roi_boxes), ), dtype=int)
    
    overlap_matrix: np.ndarray = overlap_batch(bboxes1=crop_corners, bboxes2=roi_boxes, ordered = True)
    return np.argmax(overlap_matrix, axis = 0)
    

In [ ]:
def create_training_embeddings(dataset_path: str, 
                               output_base_folder: str, 
                               percentage_to_expand_bbox_boundaries: float, 
                               classifier_label_map: Dict[int, Tuple[str, str]], 
                               mask_rcnn_model_class: MaskRCNNInstanceSegmentation, 
                               max_larger_side: int, 
                               max_smaller_side: int, 
                               crop_corners: List[List[int]], 
                               train: bool=True):
    
    reverse_label_map: Dict[Tuple[str, str], int] = {v: k for k, v in classifier_label_map.items()}
    
    dataset_name: str = os.path.basename(dataset_path)
    dataset_cell_type: str = extract_cell_type(dataset_name)

    print(f"[INFO] Processing {dataset_name} ...")
    print(f"[INFO] Cell type in the dataset: {dataset_cell_type}")
    if dataset_cell_type.lower() == 'unknown':
        print("[ERROR] The cell type in the dataset is Unknown! Cannot proceed further! Existing ... ")
    
    dataset = create_dataset_classes(dataset_path=dataset_path, 
                                     class_names_to_class_ids_map=ANNOTATIONS_CLASS_NAMES_TO_CLASS_IDS_MAP, 
                                     train=train,
                                     percentage_to_expand_bbox_boundaries=percentage_to_expand_bbox_boundaries, 
                                     max_larger_side=max_larger_side, 
                                     max_smaller_side=max_smaller_side)
    # save the results in the output folders
    if train:
        output_folder: str = os.path.join(output_base_folder, 'train')
    else:
        output_folder: str = os.path.join(output_base_folder, 'test')
            
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)

    num_objs_all = 0
    
    for idx in range(len(dataset)):
        # read the sample 
        sample = dataset[idx]

        img: np.ndarray = sample['image']

        img_height, img_width = img.shape[:2]

        # list of images cropped from the original input image and according to the passed crop_corners for cropping
        img_list: List[np.ndarray] = []
        # list of annotation bounding boxes lie in each cropped sub-image
        roi_boxes: List[np.ndarray] = []
        # list of object classes for annotations lie in each cropped sub-image
        roi_labels: List[int] = []
        # all annotations
        annotations_boxes: np.ndarray = sample['annotations'][['xtl', 'ytl', 'xbr', 'ybr']].values
        annotations_labels: np.ndarray = sample['annotations']['label'].values

        # assign the annotation bounding boxes to a crop from the list of crops
        crop_idxs: np.ndarray = assign_boxes_to_crops(roi_boxes=annotations_boxes, crop_corners=crop_corners)

        for crop_idx, crop in enumerate(crop_corners):
            img_list.append(img[crop[1]:crop[3], crop[0]: crop[2]].copy())
            filtered_roi_boxes: np.ndarray = np.array([box for i, box in enumerate(annotations_boxes) if crop_idxs[i] == crop_idx])
            filtered_roi_labels: np.ndarray = np.array([label for i, label in enumerate(annotations_labels) if crop_idxs[i] == crop_idx])
            roi_boxes.append(filtered_roi_boxes)
            roi_labels.append(filtered_roi_labels)

        # extract the embeddings
        with torch.no_grad():
            embeddings = mask_rcnn_model_class.extract_embeddings(img_list=img_list, roi_boxes=roi_boxes)
        
        num_channels: int = 1
        if len(img.shape) > 2:
            num_channels = img.shape[2]

        
        img_name: str = '.'.join(sample['name'].strip().split('.')[:-1])
        obj_count: int = 0

        for crop_idx, crop_embeddings in enumerate(embeddings):
            for obj_idx, embedding in enumerate(crop_embeddings):
                label = roi_labels[crop_idx][obj_idx]
            

                if label in ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP:
                    class_id: int = reverse_label_map[(ANNOTATIONS_NON_CELL_CLASS_NAMES_TO_IDS_MAP[label], 'na')]
                else:
                    class_id: int = reverse_label_map[(dataset_cell_type, ANNOTATIONS_CLASS_IDS_TO_CELL_STATE_MAP[label])]

            
                
                output_class_folder: str = os.path.join(output_folder, str(class_id))
                if not os.path.exists(output_class_folder):
                    os.mkdir(output_class_folder)

                # no need to detach() as we run the model with torch.no_grad()
                torch.save(embedding.cpu(), os.path.join(output_class_folder, img_name + '_' + str(obj_count) + '.pt'))

                obj_count += 1
        
        num_objs_all += obj_count
    print(f"[INFO] Total number of objects in the dataset: {num_objs_all}")

In [ ]:
### SKIP this step if training embeddings are already created
for idx, dataset_path in enumerate(DATASET_PATHS):
    # the size of images in the dataset
    # NOTE: dataset with mixed image sizes is not supported
    dataset_size: Tuple[int, int] = DATASET_SIZES[idx]
    max_larger_side: int = max(RESIZE[dataset_size])
    min_larger_side: int = min(RESIZE[dataset_size])
    crop_corners: List[List[int]] = CROP_CORNERS[dataset_size]
    
    create_training_embeddings(dataset_path=dataset_path, 
                           output_base_folder=OUTPUT_FOLDER, 
                           percentage_to_expand_bbox_boundaries = 0.1,
                           classifier_label_map=CLASSIFIER_LABEL_MAP, 
                           mask_rcnn_model_class=detector, 
                           max_larger_side=max_larger_side, 
                           max_smaller_side=min_larger_side, 
                           crop_corners=crop_corners, 
                           train=True)
    create_training_embeddings(dataset_path=dataset_path, 
                           output_base_folder=OUTPUT_FOLDER, 
                           percentage_to_expand_bbox_boundaries = 0.1,
                           classifier_label_map=CLASSIFIER_LABEL_MAP, 
                           mask_rcnn_model_class=detector, 
                           max_larger_side=max_larger_side, 
                           max_smaller_side=min_larger_side, 
                           crop_corners=crop_corners, 
                           train=False)

# Training a Classifier

In [ ]:
import torch, torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time

import random
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image

## Configurations

In [ ]:
# training batch size
BATCH_SIZE = 32
# learning rate
LEARNING_RATE = 1e-3
# number of training epochs
NUM_EPOCHS = 10
# learning rate decay steps
LR_DECAY_STEPS = 0

## Dataset model - images

In [ ]:
class CellDataset(Dataset):
    """ Dataset of cells of different types (classes) """

    def __init__(self, class_images_dict: Dict[int, List[str]], transform=None):
        """
        Args:
            class_images_dict (dictionary): A dictionary with keys as class 
                IDs (integers 0 to num_classes; 0 reserved for background if exists) and 
                values as the full path to the list of train/test images for the class ID 
                (the name should include the full path to the image).
            transform (callable, optional): Optional transform to be applied
                on a sample
        """
        
        self.image_names: List[str] = []
        self.labels: List[int] = []
        for label, image_paths_list in class_images_dict.items():
            self.image_names = self.image_names + image_paths_list
            self.labels = self.labels + [label] * len(image_paths_list)
        self.transform = transform
        self.num_classes = len(class_images_dict)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        
        image: np.ndarray = cv2.imread(self.image_names[idx], cv2.IMREAD_UNCHANGED)
        
        if np.ndim(image) == 2:
            # for gray scale channel, make them a 3-D image expected by the model
            image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
        
        # image_tensor: torch.tensor = torchvision.transforms.functional.to_tensor(image)
        
        if self.transform:
            image = self.transform(image)

        
        return image, self.labels[idx]

## Dataset model - embeddings

In [ ]:
class CellEmbeddingDataset(Dataset):
    """ Dataset of blood cells of different types (classes) """

    def __init__(self, class_embeddings_dict: Dict[int, List[str]]):
        """
        Args:
            class_images_dict (dictionary): A dictionary with keys as class 
                IDs (integers 0 to num_classes; 0 reserved for background if exists) and 
                values as the full path to the list of train/test images for the class ID 
                (the name should include the full path to the image).
            Note: This class does not accept any transforms. If needed, the transforms should
                be applied on the input images before extracting the embeddings.
        """
        
        self.embedding_filenames: List[str] = []
        self.labels: List[int] = []
        for label, embedding_paths_list in class_embeddings_dict.items():
            self.embedding_filenames = self.embedding_filenames + embedding_paths_list
            self.labels = self.labels + [label] * len(embedding_paths_list)
        
        self.num_classes = len(class_embeddings_dict)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        
        embeddings: torch.tensor = torch.load(self.embedding_filenames[idx])
        
        return embeddings, self.labels[idx]

## Data transform
### ResNet50 model

In [ ]:
# train and test data transforms
image_transforms = { 
    'train': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.RandomApply(torch.nn.ModuleList([
            torchvision.transforms.RandomRotation(degrees=[90.0, 90.0])
        ]), p=0.25),
        torchvision.transforms.RandomHorizontalFlip(p=0.5),
        torchvision.transforms.RandomVerticalFlip(p=0.5),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ]),
    'test': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ])
}

transform_image = T.Compose([T.ToTensor(), 
                             T.Resize(244), 
                             T.CenterCrop(224), 
                             T.Normalize([0.5], # these should be updated to the mean and variance of the dataset
                                         [0.5])])

### DINOv2 model

In [ ]:
# train and test data transforms
image_transforms = { 
    'train': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize(84), 
        # torchvision.transforms.CenterCrop(84), 
        torchvision.transforms.RandomApply(torch.nn.ModuleList([
            torchvision.transforms.RandomRotation(degrees=[90.0, 90.0])
        ]), p=0.25),
        torchvision.transforms.RandomHorizontalFlip(p=0.5),
        torchvision.transforms.RandomVerticalFlip(p=0.5),
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ]),
    'test': torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize(84), 
        # torchvision.transforms.CenterCrop(84), 
        torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                         [0.229, 0.224, 0.225])
    ])
}

In [ ]:
class_images_dict_test: Dict[int, List[str]] = {}
class_images_dict_train: Dict[int, List[str]] = {}

for class_id, (cell_type, cell_state) in CLASSIFIER_LABEL_MAP.items():
    image_files = os.listdir(os.path.join(OUTPUT_FOLDER, 'train', str(class_id)))
    class_images_dict_train[class_id] = [os.path.join(OUTPUT_FOLDER, 'train', str(class_id), f) for f in image_files]
    print(f"[INFO] Found {len(image_files)} images of class '{cell_type + '-' + cell_state}' with ID {class_id} for training")
    image_files = os.listdir(os.path.join(OUTPUT_FOLDER, 'test', str(class_id)))
    class_images_dict_test[class_id] = [os.path.join(OUTPUT_FOLDER, 'test', str(class_id), f) for f in image_files]
    print(f"[INFO] Found {len(image_files)} images of class '{cell_type + '-' + cell_state}' with ID {class_id} for testing")
    if class_id == 7:
        break
   

## Datasets and dataloaders - images

In [ ]:
train_dataset = CellDataset(class_images_dict_train, image_transforms['train'])
test_dataset = CellDataset(class_images_dict_test, image_transforms['test'])

train_data_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
test_data_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

## Datasets and dataloaders - embeddings

In [ ]:
train_dataset = CellEmbeddingDataset(class_images_dict_train)
test_dataset = CellEmbeddingDataset(class_images_dict_test)

train_data_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True)
test_data_loader = DataLoader(test_dataset, batch_size = 1, shuffle = False)

### Visualize some samples

In [ ]:
def show_sample_batch(sample_batch):
    """ Show training images for a batch """
    images_batch, labels_batch = sample_batch
    batch_size: int = len(labels_batch)
    grid_image = torchvision.utils.make_grid(images_batch, normalize = True)
    plt.imshow(grid_image.numpy().transpose((1, 2, 0)))
    print('Labels:' + ' '.join('%5s' % labels_batch[j].item() for j in range(batch_size)))

In [ ]:
data_iter = iter(DataLoader(train_dataset, batch_size = 4, shuffle = True))
show_sample_batch(next(data_iter))     

## Model definition
We use a ResNet50, DINOv2 (or other backbones) for the classifier using images, after replacing the linear head.

In [ ]:
def build_model(num_classes: int, freeze_backbone: bool = False):
    # get the pre-trained ResNet50 model
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
    if freeze_backbone:
        # freeze all model parameters (except the FC layer that will be replaced)
        for param in model.parameters():
            param.requires_grad_(False)
    
    # change/replace the final layer of ResNet50 model for Transfer Learning
    num_fc_inputs: int = model.fc.in_features
    """
    model.fc = nn.Sequential(
        nn.Linear(num_fc_inputs, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes), # nuumber of output classes
        )
    
    """
    model.fc = nn.Linear(num_fc_inputs, num_classes)
    
    return model


def build_dinov2_model(num_classes: int, model_type: str = "small", with_registers: bool=False):
    class Dinov2Model(nn.Module):
        def __init__(self, num_classes: int, model_type: str = "small", with_registers: bool=True) -> None:
            super(Dinov2Model, self).__init__()
            # get the pre-trained DINOv2 model based on the passed size for the model
            model_type_map = {"small": "dinov2_vits14", 
                              "base":  "dinov2_vitb14", 
                              "large": "dinov2_vitl14", 
                              "giant": "dinov2_vitg14",
                             }
            if model_type in model_type_map:
                model_type_str: str = model_type_map[model_type]
                
            else:
                model_type_str: str = "dinov2_vitb14"
                print(f"[ERROR] Incorrect model type passed {model_type}! Using the base model by default.")
        
            # DINOv2 with registers
            if with_registers:
                model_type_str += "_reg"
            
            self.dinov2 = torch.hub.load("facebookresearch/dinov2", model_type_str)
            # freeze the model
            for param in self.dinov2.parameters():
                param.requires_grad_(False)

            self.num_classes = num_classes
            # the dimension of the CLS embeddings
            self.num_fc_inputs: int = self.dinov2.norm.normalized_shape[0]
            self.fc = nn.Linear(self.num_fc_inputs, self.num_classes)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.fc(self.dinov2(x))
    
    return Dinov2Model(num_classes=num_classes, model_type=model_type, with_registers=with_registers)

def build_embeddings_model(num_classes: int):
    # a two layer head
    class EmbeddingsModel(nn.Module):
        def __init__(self, num_classes: int, roi_feature_dim: int = 1024, dropout: float = 0.4) -> None:
            super(EmbeddingsModel, self).__init__()
            self.num_classes: int = num_classes
            self.roi_feature_dim: int = roi_feature_dim
            self.fc = nn.Linear(self.roi_feature_dim, self.num_classes)
    
        def forward(self, embedding: torch.Tensor) -> torch.Tensor:
            return self.fc(embedding)
    
    return EmbeddingsModel(num_classes=num_classes)

## Training
### Optimizer settings

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Device available:' , device)

# model = build_model(num_classes=8, freeze_backbone=False)
model = build_dinov2_model(num_classes=8, model_type= "large", with_registers=True)
# model = build_embeddings_model(num_classes=8)
params = [p for p in model.parameters() if p.requires_grad]

# move model to the right device
model.train()
model.to(device)

# construct an optimizer

optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)
print('Adam Optimizer is configured for %d epochs' %NUM_EPOCHS)

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_data_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_data_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)


criterion = nn.CrossEntropyLoss()

### Training script

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          num_epochs,
          device):
    
    torch.cuda.empty_cache()
    
    # losses, accuracies and mean IoUs over the training epochs
    train_losses: List[float] = []
    test_losses: List[float] = []
    train_accs: List[float] = []
    test_accs: List[float] = []
    test_precision: List[float] = []
    test_recall: List[float] = []
    
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []
    min_loss: float = np.inf
    
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        accuracy: float = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            image_tensors, label_tensors = data
            
            image_tensors = image_tensors.to(device) 
            label_tensors = label_tensors.to(device)
            
            # forward
            output = model(image_tensors)
            loss = criterion(output, label_tensors)
            # evaluate metrics
            # compute the accuracy
            
            _, predictions = torch.max(output.data, dim=1)
            correct = predictions.eq(label_tensors.data.view_as(predictions)).int()
            
            # convert correct_counts to float and then compute the mean
            accuracy += float(correct.sum()) / float(correct.numel())
            
            # backward
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 
            
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 
        
        # run the validation after each training epoch
        model.eval()
        test_running_loss: float = 0
        test_accuracy: float = 0
        
        # validation loop
        with torch.no_grad():
            for i, data in enumerate(tqdm(test_loader)):
                
                image_tensors, label_tensors = data
            
                image_tensors = image_tensors.to(device) 
                label_tensors = label_tensors.to(device)

                output = model(image_tensors)
                # evaluation metrics
                # compute the accuracy
                
                _, predictions = torch.max(output.data, dim=1)
                correct = predictions.eq(label_tensors.data.view_as(predictions)).int()
                
                # convert correct_counts to float and then compute the mean
                test_accuracy += float(correct.sum()) / float(correct.numel())
                # loss
                loss = criterion(output, label_tensors)                                  
                test_running_loss += loss.item()
            
        # calculatio mean for each batch
        running_loss /= len(train_loader)
        accuracy /= len(train_loader)
        
        test_running_loss /= len(test_loader)
        test_accuracy /= len(test_loader)
                       
         # save the results
        train_losses.append(running_loss)
        train_accs.append(accuracy)
        
        test_losses.append(test_running_loss)
        test_accs.append(test_accuracy)
        
        print('saving the model ...')
        torch.save(model.state_dict(), os.path.join('classifier_models', 'checkpoint_' + str(epoch) +'.pt'))
                    
        
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Test Loss: {:.3f} \n".format(test_running_loss),
              "Train Accuracy: {:.3f} \n".format(accuracy),
              "Test Accuracy: {:.3f} \n".format(test_accuracy),
              "Time: {:.2f} m".format((time.time() - since) / 60))
        
    history = {'train_loss' : train_losses, 'test_loss': test_losses,
               'train_acc': train_accs, 'val_acc': test_accs,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))
    return history

In [ ]:
history = train(model, 
          train_data_loader, 
          test_data_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          NUM_EPOCHS,
          device)

## Evaluation
### Confusion matrix

In [ ]:
# load the best model_dict from file after training 

model = build_dinov2_model(num_classes=8, model_type= "large", with_registers=True)
model.load_state_dict(torch.load(os.path.join('classifier_models', 'cell_classifier_frozen_dinov2_bb_1_layer.pt')))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

In [ ]:
label_map = {0: 'jurkat',
             1: 'k562',
             2: 'nk92',
             3: 'imr90',
             4: 'hela',
             5: 'mousepbmc',
             6: 'pbmc',
             7: 'bead'}

In [ ]:
def predict(model, image_tensor):
    image_tensor: torch.tensor = image_tensor.unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():   
        # model outputs log probabilities
        out = model(image_tensor)
        # total = torch.exp(out).sum().item()
        # topk, topclass = out.topk(3, dim=1)
        # for i in range(3):
        #     print("Predcition", i + 1, ":", topclass.cpu().numpy()[0][i], ", Score: ", np.exp(topk.cpu().numpy()[0][i])/total)
        ret, prediction = torch.max(out.data, 1)
        return prediction.item()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score 
import seaborn as sn

y_pred = []
y_true = []

# iterate over test data
for image_tensors, label_tensors in test_data_loader:
    image_tensors = image_tensors.to(device)
    label_tensors = label_tensors.to(device)
    with torch.no_grad():
        outputs = model(image_tensors) # Feed Network

        predictions = (torch.max(outputs.data, dim=1)[1]).data.cpu().numpy()
        y_pred.extend(predictions) # Save Prediction
        
        labels = label_tensors.data.cpu().numpy()
        y_true.extend(labels) 

# Build confusion matrix
cf_matrix = confusion_matrix(y_true, y_pred)
df_cm = pd.DataFrame(cf_matrix / np.sum(cf_matrix, axis=1)[:, None], index = list(label_map.values()),
                     columns = list(label_map.values()))
plt.figure(figsize = (12,7))
sn_plot = sn.heatmap(df_cm, annot=True)
print(f"Accuracy: {np.round(accuracy_score(y_pred, y_true) * 100, 2)}%")

In [ ]:
# save to file
sn_plot.get_figure().savefig("classifier_c_m_frozen_dinov2_bb_1_layer.png")

## Extract emebeddings of the test set

In [ ]:
# replace the fully connected layer with an identity layer
model.fc = nn.Identity()
model.to(device)
model.eval()

In [ ]:
embeddings = []
labels = []
for data in test_data_loader:
    image_tensors, label_tensors = data
    image_tensors = image_tensors.to(device)
    label_tensors = label_tensors.to(device)    
   
    with torch.no_grad():
         features = model(image_tensors)
    embeddings.append(features.cpu())
    labels.append(label_tensors.cpu())
embeddings = torch.cat(embeddings, dim = 0)
labels = torch.cat(labels, dim = 0)

In [ ]:
# convert to numpy before any dimentionality reduction
embeddings = embeddings.numpy()
labels = labels.numpy()

## T-SNE visualization

In [ ]:
from sklearn.manifold import TSNE
import seaborn as sn
# we want to get T-SNE embedding with 2 dimensions
n_components = 2
tsne = TSNE(n_components)
tsne_result = tsne.fit_transform(embeddings)
# Plot the result of our TSNE with the label color coded
# A lot of the stuff here is about making the plot look pretty and not TSNE
# pick the same number of samples from each class for display (to make the distribution more clear)
unique_labels = np.unique(labels)
class_idxs = {}
MIN_NUM_REQUIRED_SAMPLES = 1000
num_samples = 1e9
for label in unique_labels:
    idxs = np.where(labels == label)[0]
    np.random.shuffle(idxs)
    class_idxs[label] = idxs
    if len(idxs) < MIN_NUM_REQUIRED_SAMPLES:
        continue
    num_samples = min(num_samples, len(idxs))
    
idxs_to_use = np.hstack([c_idxs[:num_samples] for _, c_idxs in class_idxs.items()])

tsne_result_df = pd.DataFrame({'tsne_1': tsne_result[idxs_to_use,0], 'tsne_2': tsne_result[idxs_to_use,1], 'label': labels[idxs_to_use]})
tsne_result_df['label'] = tsne_result_df['label'].map(label_map)
fig, ax = plt.subplots(1)
sn_plot = sn.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df, ax=ax,s=120)
lim = (tsne_result.min()-5, tsne_result.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
sn_plot.get_figure().savefig("classifier_t_sne_frozen_dinov2_bb_1_layer.png")

## UMAP visualization

In [ ]:
import umap
# UMAP embeddings will be with 2 dimensions
reducer = umap.UMAP()
umap_result = reducer.fit_transform(embeddings)
# Plot the result of our UMAP with the label color coded
unique_labels = np.unique(labels)
class_idxs = {}
MIN_NUM_REQUIRED_SAMPLES = 1000
num_samples = 1e9
for label in unique_labels:
    idxs = np.where(labels == label)[0]
    np.random.shuffle(idxs)
    class_idxs[label] = idxs
    if len(idxs) < MIN_NUM_REQUIRED_SAMPLES:
        continue
    num_samples = min(num_samples, len(idxs))
    
idxs_to_use = np.hstack([c_idxs[:num_samples] for _, c_idxs in class_idxs.items()])
umap_result_df = pd.DataFrame({'umap_1': umap_result[idxs_to_use, 0], 'umap_2': umap_result[idxs_to_use, 1], 'label': labels[idxs_to_use]})
umap_result_df['label'] = umap_result_df['label'].map(label_map)
fig, ax = plt.subplots(1)
sn_plot = sn.scatterplot(x='umap_1', y='umap_2', hue='label', data=umap_result_df, ax=ax,s=120)
lim = (umap_result.min()-1, umap_result.max()+1)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
sn_plot.get_figure().savefig("classifier_umap_frozen_dinov2_bb_1_layer.png")

12.0